# MCP using local servers

This notebook shows how an LLM can **chain** several MCP tools to finish a task, and how to run an MCP server locally and connect to it over HTTP.

## Learning Objectives

At the end of this notebook, you should be able to:

- Implement an agent loop that lets an LLM call MCP tools over several turns.
- Write a system prompt that guides the model through a multi-step task.
- Run a local MCP server over HTTP and connect a client to it.
- Build a new MCP tool of your own and call it from the agent loop.

## Why Chain Tools?

Many real tasks need more than one step: read a file, run what it contains, then explain the output. A single tool call cannot do that, because the model must see the result of each step before it chooses the next.

The solution is a **loop**: call the model, run the tool it asks for, add the result to the conversation, and repeat until the model answers without asking for a tool. The alternative is to hard-code the sequence of steps yourself. That is predictable, but it only works for the exact task you planned. With a loop the model decides the steps, which is more flexible but less reliable, so the loop below has a limit of 5 turns.

## The Tools

The file `mcp_servers/mcp_execute_python_server.py` offers two MCP tools:

1. `read_local_file`: the same tool as in [1_intro_to_MCP](1_intro_to_MCP.ipynb).
2. `execute_python_code`: a new tool that runs Python code directly.

We combine both tools and use **Chain of Thought** prompting, where the model works through the task step by step:

1. Read a Python script from your machine with `read_local_file`.
2. Send what it read to `execute_python_code`.
3. Use the output to answer.

> **Warning:** `execute_python_code` runs whatever code the model sends, on your machine, with a 10-second time limit. Only run this server locally, and never expose it to a network you do not control.

In [2]:
import ollama
from mcp import ClientSession
from mcp.client.sse import sse_client as http_client

In [3]:
LOCAL_SERVER_URL = "http://127.0.0.1:54321/sse"

## The System Prompt and the Agent Loop

The **system prompt** tells the model how to behave on every turn: work step by step, and feed the output of one tool into the next. The function `run_chainable_agent` then runs the loop for at most 5 turns:

1. Ask the model what to do next, giving it the conversation so far and the list of tools.
2. If it asks for tools, call each one through the MCP session and add the result to the conversation.
3. If it answers without asking for a tool, return that answer.

The turn limit stops a confused model from looping forever.

In [4]:
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. "
    "If a task requires multiple steps (e.g., reading a file then running it), "
    "perform them one by one. Use the output of one tool to inform the next. "
    "If you need to read a file, use the file reading tool first. If you need to execute code, use the code execution tool."
    "After getting result from executing code, give the output as the final answer."
    "When you have the final answer, summarize it for the user."
    "If the python files have syntax error then output the syntax error as the final answer without trying to execute it and tell what the error is"
)

In [5]:
async def run_chainable_agent(user_prompt, system_prompt, model):
    # Define the conversation history
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    async with http_client(LOCAL_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 1. Fetch and format tools for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.input_schema,
                    },
                }
                for t in tools_result.tools
            ]

            print(f"🚀 Starting agent loop for: '{user_prompt}'")

            # 2. Start the Loop (Max 5 turns to prevent infinite loops)
            for turn in range(5):
                print(f"\n--- Turn {turn + 1} ---")

                response = ollama.chat(
                    model=model, messages=messages, tools=ollama_tools
                )

                assistant_msg = response["message"]
                messages.append(assistant_msg)  # Record the model's reasoning/call

                # Check if the model wants to call tools
                if not assistant_msg.get("tool_calls"):
                    print("✅ Final answer reached.")
                    return assistant_msg["content"]

                # 3. Execute Tool Calls
                for tool_call in assistant_msg["tool_calls"]:
                    t_name = tool_call["function"]["name"]
                    t_args = tool_call["function"]["arguments"]

                    print(f"🛠️ Tool Call: {t_name}({t_args})")

                    # Execute the tool via MCP
                    result = await session.call_tool(t_name, t_args)

                    # Append the result back to the conversation
                    messages.append(
                        {"role": "tool", "content": str(result.content), "name": t_name}
                    )
                    print("📥 Tool Result received.")

            return "⚠️ Loop limit reached without a final answer."

In [6]:
query = "Read fibonacci_numbers.py in the scripts folder and then execute it. Path to file=scripts/fibonacci_numbers.py"
MODEL = "llama3.2:3b"

result = await run_chainable_agent(query, SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

🚀 Starting agent loop for: 'Read fibonacci_numbers.py in the scripts folder and then execute it. Path to file=scripts/fibonacci_numbers.py'

--- Turn 1 ---
🛠️ Tool Call: execute_python_code({'file_path': 'scripts/fibonacci_numbers.py'})
📥 Tool Result received.

--- Turn 2 ---
✅ Final answer reached.

RESULT:
The output of the Python code in fibonacci_numbers.py is:

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

This is the Fibonacci sequence up to the 10th number.


## Running the Local Server

When you run the cell above, you get an error because the MCP server is not running yet. To start it:

1. Open a terminal.
2. Navigate to this repository.
3. Run `uv run python mcp_servers/mcp_execute_python_server.py`.

After you run this command, the server starts and listens for connections.

### Check That the Server Is Listening

1. Open another terminal and navigate to the repository.
2. Run `curl -v http://127.0.0.1:54321/sse`.
3. If the connection works, you see `HTTP/1.1 200 OK` in this terminal.
4. In the terminal where the server runs, you also see `200 OK`.
5. Stop `curl` with `control + c`.

### Connect the LLM to the MCP Server

Run the cell above again to see whether the client can connect to the server.

### Reading the Results

These observations come from one run with `llama3.2:3b`. Your run may differ.

- **Turn 1.** The model called `execute_python_code` once, with `file_path` set to `scripts/fibonacci_numbers.py`. It did not call `read_local_file` first, because the execute tool can run a file directly.
- **Turn 2.** With the tool result in the conversation, the model gave its final answer and listed the first ten Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13, 21 and 34.

So the loop finished in two turns, not five, and the model skipped the separate read step. The loop is ready for longer chains: if the model had asked to read the file first, the next turn would have received that content before the next tool call.

## DIY - Do It Yourself

Now try to implement new mcp tools and make them work for you. You can start with very simple math tools.


### Step 1: Add Math Tools to the Server

I added three simple tools to `mcp_servers/mcp_execute_python_server.py`: `add`, `multiply` and `power`. Each is a plain Python function with the `@mcp.tool()` decorator. The type hints and the docstring tell the model what the tool does and which arguments it needs.

**Restart the server** so it picks up the new tools: stop it with `control + c` in its terminal, then run `uv run python mcp_servers/mcp_execute_python_server.py` again.

### Step 2: Chain the Tools

The question below needs two tools in a row: `add` first, then `multiply` with the result of `add`. This is the same loop as before, only with a math prompt.

In [7]:
MATH_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to math tools. "
    "Never calculate in your head. Use the tools for every calculation, one step at a time, "
    "and use the result of one tool as input for the next. "
    "When you have the final number, state it clearly."
)

math_query = "What is (3 + 4) multiplied by 5?"

result = await run_chainable_agent(math_query, MATH_SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

🚀 Starting agent loop for: 'What is (3 + 4) multiplied by 5?'

--- Turn 1 ---
🛠️ Tool Call: execute_python_code({'code': '(3 + 4) * 5'})
📥 Tool Result received.

--- Turn 2 ---
✅ Final answer reached.

RESULT:
The result of (3 + 4) is 7. 

7 multiplied by 5 is 35.


## Summary

In this notebook you:

- Implemented an agent loop that let an LLM call MCP tools over several turns, with a limit of 5.
- Wrote a system prompt that guided the model through a multi-step task.
- Ran a local MCP server over HTTP and connected a client to it.
- Practised building a tool of your own in the DIY section.

## References & Further Reading

- [**Model Context Protocol: Transports**](https://modelcontextprotocol.io/docs/concepts/transports): How clients and servers talk to each other, including SSE over HTTP
- [**MCP Python SDK**](https://py.sdk.modelcontextprotocol.io/): Documentation for building servers and clients
- [**Ollama Tool Calling**](https://docs.ollama.com/capabilities/tool-calling): How a local model chooses and calls tools
- [**Chain-of-Thought Prompting**](https://arxiv.org/abs/2201.11903): The research paper behind step-by-step prompting